# 🎬 ROTBOTS — Video Plan

---

Configure your video: length, content mix, and features.

This plan controls the entire pipeline — all other notebooks read it.

> **🤔** You're the director. The machine builds what you specify.

---

In [ ]:
# SETUP
!pip install -q requests beautifulsoup4 pymupdf
import json, re
from pathlib import Path
from IPython.display import display, Markdown

from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
print('✅ Setup complete')

---
## 🎯 Your Video

In [ ]:
# ============================================================
# VIDEO PLAN — Configure everything here
# ============================================================

TOPIC = 'The history and ethics of AI-generated art'

SOURCES = [
    'https://en.wikipedia.org/wiki/Artificial_intelligence_art',
]

# LENGTH & MIX
TOTAL_VIDEO_LENGTH = 60
SECONDS_PER_SCENE = 5
ARCHIVE_RATIO = 0.0
UPLOAD_RATIO = 0.0

# FEATURES
ENABLE_CREDITS = True
ENABLE_SUBTITLES = False
ENABLE_MUSIC = False
ENABLE_EFFECTS = True

SESSION_NAME = ''

In [ ]:
# CALCULATE PLAN
AI_RATIO = max(0, 1.0 - ARCHIVE_RATIO - UPLOAD_RATIO)
CREDITS_LENGTH = 8 if ENABLE_CREDITS else 0
CONTENT_LENGTH = TOTAL_VIDEO_LENGTH - CREDITS_LENGTH
AI_LENGTH = CONTENT_LENGTH * AI_RATIO
ARCHIVE_LENGTH = CONTENT_LENGTH * ARCHIVE_RATIO
UPLOAD_LENGTH = CONTENT_LENGTH * UPLOAD_RATIO
NARRATION_LENGTH = CONTENT_LENGTH
NUM_TOTAL_SCENES = max(2, int(CONTENT_LENGTH / SECONDS_PER_SCENE))
NUM_AI_SCENES = max(1, int(NUM_TOTAL_SCENES * AI_RATIO))
NUM_ARCHIVE_SCENES = int(NUM_TOTAL_SCENES * ARCHIVE_RATIO)
NUM_UPLOAD_SCENES = NUM_TOTAL_SCENES - NUM_AI_SCENES - NUM_ARCHIVE_SCENES

scene_types = []
ai_p=0;ar_p=0;up_p=0
for i in range(NUM_TOTAL_SCENES):
    ai_d=(i+1)*AI_RATIO-ai_p; ar_d=(i+1)*ARCHIVE_RATIO-ar_p; up_d=(i+1)*UPLOAD_RATIO-up_p
    if ar_d>=ai_d and ar_d>=up_d and ar_p<NUM_ARCHIVE_SCENES: scene_types.append('archive');ar_p+=1
    elif up_d>=ai_d and up_p<NUM_UPLOAD_SCENES: scene_types.append('upload');up_p+=1
    else: scene_types.append('ai_generated');ai_p+=1

if not SESSION_NAME:
    words = re.sub(r'[^a-zA-Z0-9\s]','',TOPIC.lower()).split()
    SESSION_NAME = '-'.join(words[:4])
SESSION_DIR = BASE_DIR / SESSION_NAME
SESSION_DIR.mkdir(parents=True, exist_ok=True)
(SESSION_DIR/'videos').mkdir(exist_ok=True)
(SESSION_DIR/'audio').mkdir(exist_ok=True)

plan = {'topic':TOPIC,'sources':SOURCES,'session_name':SESSION_NAME,'total_video_length':TOTAL_VIDEO_LENGTH,'seconds_per_scene':SECONDS_PER_SCENE,'content_length':CONTENT_LENGTH,'credits_length':CREDITS_LENGTH,'narration_length':NARRATION_LENGTH,'ai_ratio':AI_RATIO,'archive_ratio':ARCHIVE_RATIO,'upload_ratio':UPLOAD_RATIO,'ai_length':round(AI_LENGTH,1),'archive_length':round(ARCHIVE_LENGTH,1),'upload_length':round(UPLOAD_LENGTH,1),'num_total_scenes':NUM_TOTAL_SCENES,'num_ai_scenes':NUM_AI_SCENES,'num_archive_scenes':NUM_ARCHIVE_SCENES,'num_upload_scenes':NUM_UPLOAD_SCENES,'scene_types':scene_types,'enable_credits':ENABLE_CREDITS,'enable_subtitles':ENABLE_SUBTITLES,'enable_music':ENABLE_MUSIC,'enable_effects':ENABLE_EFFECTS}
with open(SESSION_DIR/'video_plan.json','w') as f: json.dump(plan,f,indent=2)
with open(SESSION_DIR/'session_info.json','w') as f: json.dump({'name':SESSION_NAME,'topic':TOPIC,'target_length':TOTAL_VIDEO_LENGTH},f,indent=2)

print(f'📂 Session: {SESSION_NAME}')
print(f'🎬 Total: {TOTAL_VIDEO_LENGTH}s = {CONTENT_LENGTH}s content + {CREDITS_LENGTH}s credits')
print(f'   🤖 AI: {AI_RATIO:.0%} = {AI_LENGTH:.0f}s ({NUM_AI_SCENES} scenes)')
print(f'   🏛️ Archive: {ARCHIVE_RATIO:.0%} = {ARCHIVE_LENGTH:.0f}s ({NUM_ARCHIVE_SCENES} scenes)')
print(f'   📂 Upload: {UPLOAD_RATIO:.0%} = {UPLOAD_LENGTH:.0f}s ({NUM_UPLOAD_SCENES} scenes)')
icons = ['🤖' if t=='ai_generated' else '🏛️' if t=='archive' else '📂' for t in scene_types]
print(f'   Order: {" → ".join(icons)}')
print(f'\n✅ Plan saved')

---
*ROTBOTS Workshop — LI-MA 2026*